In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [3]:
# Cargar las tablas principales
patients   = pd.read_csv('./datasets/mimic-iii-clinical-database/PATIENTS.csv')
admissions = pd.read_csv('./datasets/mimic-iii-clinical-database/ADMISSIONS.csv')
icustays   = pd.read_csv('./datasets/mimic-iii-clinical-database/ICUSTAYS.csv')

# Verificar dimensiones de cada tabla
print("PATIENTS:")
print(f"  Filas: {patients.shape[0]}, Columnas: {patients.shape[1]}")
print(f"  Columnas: {patients.columns.tolist()}\n")

print("ADMISSIONS:")
print(f"  Filas: {admissions.shape[0]}, Columnas: {admissions.shape[1]}")
print(f"  Columnas: {admissions.columns.tolist()}\n")

print("ICUSTAYS:")
print(f"  Filas: {icustays.shape[0]}, Columnas: {icustays.shape[1]}")
print(f"  Columnas: {icustays.columns.tolist()}\n")

# Unir las tres tablas por SUBJECT_ID y HADM_ID
df = pd.merge(admissions, patients, on='subject_id', how='left')
df = pd.merge(df, icustays, on=['subject_id', 'hadm_id'], how='left')

print(f"Dataset combinado: {df.shape}")
print(f"\nColumnas del dataset combinado:")
print(df.columns.tolist())

PATIENTS:
  Filas: 100, Columnas: 8
  Columnas: ['row_id', 'subject_id', 'gender', 'dob', 'dod', 'dod_hosp', 'dod_ssn', 'expire_flag']

ADMISSIONS:
  Filas: 129, Columnas: 19
  Columnas: ['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospital_expire_flag', 'has_chartevents_data']

ICUSTAYS:
  Filas: 136, Columnas: 12
  Columnas: ['row_id', 'subject_id', 'hadm_id', 'icustay_id', 'dbsource', 'first_careunit', 'last_careunit', 'first_wardid', 'last_wardid', 'intime', 'outtime', 'los']

Dataset combinado: (136, 36)

Columnas del dataset combinado:
['row_id_x', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospi

In [4]:
# Seleccionar solo las columnas útiles para predecir mortalidad
columnas = [
    'subject_id',          # ID del paciente
    'gender',              # género del paciente
    'admission_type',      # tipo de admisión (EMERGENCY, ELECTIVE, etc.)
    'admission_location',  # de dónde viene el paciente
    'insurance',           # tipo de seguro médico
    'marital_status',      # estado civil
    'expire_flag',# variable objetivo: 0=sobrevivió, 1=falleció
    'los',                 # duración de la estancia en UCI (days)
]

# Filtrar solo columnas que existen en el dataset
columnas_existentes = [c for c in columnas if c in df.columns]
df = df[columnas_existentes].copy()

print(f"Columnas seleccionadas: {df.columns.tolist()}")
print(f"\nShape: {df.shape}")
print(f"\nPrimeras 5 filas:")
print(df.head())

Columnas seleccionadas: ['subject_id', 'gender', 'admission_type', 'admission_location', 'insurance', 'marital_status', 'expire_flag', 'los']

Shape: (136, 8)

Primeras 5 filas:
   subject_id gender admission_type         admission_location insurance  \
0       10006      F      EMERGENCY       EMERGENCY ROOM ADMIT  Medicare   
1       10011      F      EMERGENCY  TRANSFER FROM HOSP/EXTRAM   Private   
2       10013      F      EMERGENCY  TRANSFER FROM HOSP/EXTRAM  Medicare   
3       10017      F      EMERGENCY       EMERGENCY ROOM ADMIT  Medicare   
4       10019      M      EMERGENCY  TRANSFER FROM HOSP/EXTRAM  Medicare   

  marital_status  expire_flag      los  
0      SEPARATED            1   1.6325  
1         SINGLE            1  13.8507  
2            NaN            1   2.6499  
3       DIVORCED            1   2.1436  
4       DIVORCED            1   1.2938  


In [5]:
# Convertir fechas a datetime
admissions['admittime']  = pd.to_datetime(admissions['admittime'])
admissions['dischtime']  = pd.to_datetime(admissions['dischtime'])
admissions['deathtime']  = pd.to_datetime(admissions['deathtime'])

# Calcular duración de la hospitalización en horas
admissions['duracion_horas'] = (
    admissions['dischtime'] - admissions['admittime']
).dt.total_seconds() / 3600

print(f"Duración promedio hospitalización: {admissions['duracion_horas'].mean():.1f} horas")
print(f"Duración mínima: {admissions['duracion_horas'].min():.1f} horas")
print(f"Duración máxima: {admissions['duracion_horas'].max():.1f} horas")



# Unir admissions con df usando una columna común
df = df.merge(admissions[['subject_id', 'duracion_horas']], on='subject_id', how='left')

Duración promedio hospitalización: 224.0 horas
Duración mínima: 0.9 horas
Duración máxima: 2975.6 horas


In [6]:
# Ver cuántos nulos hay por columna
nulos = df.isnull().sum()
porcentaje = (nulos / len(df) * 100).round(1)
resumen_nulos = pd.DataFrame({
    'nulos': nulos,
    'porcentaje': porcentaje
})
resumen_nulos = resumen_nulos[resumen_nulos['nulos'] > 0].sort_values('porcentaje', ascending=False)
print("Columnas con valores nulos:")
print(resumen_nulos)

# Imputar nulos en columna numérica LOS con la mediana
if 'LOS' in df.columns:
    mediana_los = df['LOS'].median()
    df['LOS'] = df['LOS'].fillna(mediana_los)
    print(f"\nMediana LOS usada para imputar: {mediana_los:.2f}")

# Imputar nulos en columnas categóricas con 'Unknown'
cols_categoricas = df.select_dtypes(include='object').columns
for col in cols_categoricas:
    df[col] = df[col].fillna('Unknown')

print(f"\nNulos restantes: {df.isnull().sum().sum()}")

Columnas con valores nulos:
                nulos  porcentaje
marital_status     18         4.7

Nulos restantes: 0


In [7]:
# Identificar tipos de variables
numericas   = df.select_dtypes(include='number').columns.tolist()
categoricas = df.select_dtypes(include='object').columns.tolist()

print(f"Variables numéricas ({len(numericas)}):")
print(numericas)
print(f"\nVariables categóricas ({len(categoricas)}):")
print(categoricas)

# Ver valores únicos de cada categórica
for col in categoricas:
    print(f"\n{col} — {df[col].nunique()} valores únicos:")
    print(df[col].value_counts())

Variables numéricas (4):
['subject_id', 'expire_flag', 'los', 'duracion_horas']

Variables categóricas (5):
['gender', 'admission_type', 'admission_location', 'insurance', 'marital_status']

gender — 2 valores únicos:
gender
M    309
F     73
Name: count, dtype: int64

admission_type — 3 valores únicos:
admission_type
EMERGENCY    370
ELECTIVE      10
URGENT         2
Name: count, dtype: int64

admission_location — 5 valores únicos:
admission_location
EMERGENCY ROOM ADMIT         183
CLINIC REFERRAL/PREMATURE    139
TRANSFER FROM HOSP/EXTRAM     32
TRANSFER FROM SKILLED NUR     15
PHYS REFERRAL/NORMAL DELI     13
Name: count, dtype: int64

insurance — 4 valores únicos:
insurance
Medicare      335
Private        37
Medicaid        9
Government      1
Name: count, dtype: int64

marital_status — 7 valores únicos:
marital_status
MARRIED              283
SINGLE                41
WIDOWED               22
Unknown               18
UNKNOWN (DEFAULT)     11
DIVORCED               6
SEPARATED    

In [14]:
# GENDER solo tiene 2 valores — reemplazar directamente
df['gender'] = df['gender'].map({'M': 0, 'F': 1})
print("gender codificado: M=0, F=1")

# ADMISSION_TYPE — pocos valores, codificación manual
print(f"\nadmission_type valores únicos:")
print(df['admission_type'].value_counts())

tipo_admision = {
    'emergency':        0,
    'elective':         1,
    'urgent':           2,
    'newborn':          3,
    'unknown':          4
}
df['admission_type'] = df['admission_type'].map(tipo_admision).fillna(4)

# INSURANCE, MARITAL_STATUS, ADMISSION_LOCATION
# tienen más valores — usar One Hot Encoding
cols_ohe = ['insurance', 'marital_status', 'admission_location']
cols_ohe_existentes = [c for c in cols_ohe if c in df.columns]

df = pd.get_dummies(df, columns=cols_ohe_existentes, drop_first=True, dtype=int)
print(f"\nShape después de One Hot Encoding: {df.shape}")
print(f"\nColumnas finales:")
print(df.columns.tolist())

gender codificado: M=0, F=1

admission_type valores únicos:
admission_type
4.0    382
Name: count, dtype: int64

Shape después de One Hot Encoding: (382, 19)

Columnas finales:
['subject_id', 'gender', 'admission_type', 'expire_flag', 'los', 'duracion_horas', 'insurance_Medicaid', 'insurance_Medicare', 'insurance_Private', 'marital_status_MARRIED', 'marital_status_SEPARATED', 'marital_status_SINGLE', 'marital_status_UNKNOWN (DEFAULT)', 'marital_status_Unknown', 'marital_status_WIDOWED', 'admission_location_EMERGENCY ROOM ADMIT', 'admission_location_PHYS REFERRAL/NORMAL DELI', 'admission_location_TRANSFER FROM HOSP/EXTRAM', 'admission_location_TRANSFER FROM SKILLED NUR']


In [18]:
# Separar variable objetivo
y = df['expire_flag'].values
X = df.drop(columns=['expire_flag', 'subject_id']).values

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Verificar tipos de datos antes de normalizar
print("Tipos de datos en el dataframe:")
print(df.dtypes)

# Ver si quedaron columnas no numéricas
no_numericas = df.select_dtypes(exclude='number').columns.tolist()
print(f"\nColumnas no numéricas que quedaron: {no_numericas}")

# Normalización
def featureNormalize(X):
    mu    = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma[sigma == 0] = 1
    return (X - mu) / sigma, mu, sigma

# Convertir booleanos y enteros a float para que NumPy sea feliz
X = X.astype(float)

# Split primero
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalizar con datos de train
X_train_norm, mu, sigma = featureNormalize(X_train)
X_test_norm  = (X_test - mu) / sigma

print(f"\nEntrenamiento: {X_train_norm.shape}")
print(f"Prueba:        {X_test_norm.shape}")
print(f"\nMedia X_train_norm (debe ser ~0): {X_train_norm.mean():.6f}")
print(f"Std X_train_norm  (debe ser ~1): {X_train_norm.std():.6f}")

X shape: (382, 17)
y shape: (382,)
Tipos de datos en el dataframe:
subject_id                                        int64
gender                                          float64
admission_type                                  float64
expire_flag                                       int64
los                                             float64
duracion_horas                                  float64
insurance_Medicaid                                 bool
insurance_Medicare                                 bool
insurance_Private                                  bool
marital_status_MARRIED                             bool
marital_status_SEPARATED                           bool
marital_status_SINGLE                              bool
marital_status_UNKNOWN (DEFAULT)                   bool
marital_status_Unknown                             bool
marital_status_WIDOWED                             bool
admission_location_EMERGENCY ROOM ADMIT            bool
admission_location_PHYS REFERRAL/NORM